# Premier League — Modelo de Previsão de Resultados

**Objetivo:** treinar e avaliar um `HistGradientBoostingClassifier` para prever o resultado de partidas da Premier League em três classes: *HomeWin*, *Draw*, *AwayWin*.

**Premissa:** todas as features já estão calculadas e prontas nos parquets gerados pelo `notebook_data.ipynb`. Este notebook não realiza nenhuma transformação de dados — apenas carrega, treina e avalia.

| Arquivo | Conteúdo |
|---------|----------|
| `dados/feature_matrix_train.parquet` | Treino — seasons 2005/06 → 2021/22 |
| `dados/feature_matrix_test.parquet` | Teste — seasons 2022/23 → 2025/26 |

> **Baseline ingênuo (sempre HomeWin):** 45,6 %

## Seção 1 — Imports

In [ ]:
import os
os.environ["MPLBACKEND"] = "agg"

import difflib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

print("Imports OK.")

## Seção 2 — Load Data

In [ ]:
train = pd.read_parquet("dados/feature_matrix_train.parquet")
test  = pd.read_parquet("dados/feature_matrix_test.parquet")

print(f"Treino : {train.shape}  |  seasons {train['season'].min()}–{train['season'].max()}")
print(f"Teste  : {test.shape}  |  seasons {test['season'].min()}–{test['season'].max()}")
train.head(3)

## Seção 3 — Definir X e y

In [ ]:
NON_FEATURE_COLS = [
    # Identificadores e metadados
    "Date", "Season", "season", "home_team", "away_team",
    # Alvo e resultado bruto
    "result", "FTR",
    # Scores brutos — leakage (não disponíveis antes do apito)
    "home_score", "away_score", "HTHG", "HTAG", "HTR",
    # Estatísticas brutas do jogo — leakage
    "HS", "AS", "HST", "AST", "HF", "AF",
    "HC", "AC", "HY", "AY", "HR", "AR", "Referee",
    # Odds brutas — já normalizadas como implied_*_prob
    "B365H", "B365D", "B365A",
    # Elo — redundante com as odds implícitas; remover desta lista para reativar
    "elo_home", "elo_away", "elo_diff",
]

FEATURE_COLS = [c for c in train.columns if c not in NON_FEATURE_COLS]

X_train = train[FEATURE_COLS]
y_train = train["result"]
X_test  = test[FEATURE_COLS]
y_test  = test["result"]

print(f"Total de features: {len(FEATURE_COLS)}")
print(FEATURE_COLS)

## Seção 4 — Recency Weighting

Jogos mais recentes recebem peso maior via decaimento exponencial por temporada. Com `α = 0.10`, a temporada mais recente pesa ~e^(17×0.1) ≈ 5.5× mais que a primeira.

In [ ]:
season_weight = np.exp(0.10 * (train["season"] - train["season"].min()))
sample_weight = (season_weight / season_weight.mean()).values

print(f"Peso mínimo (season {train['season'].min()}): {sample_weight.min():.3f}")
print(f"Peso máximo (season {train['season'].max()}): {sample_weight.max():.3f}")

## Seção 5 — Treino

`HistGradientBoostingClassifier` com early stopping interno: reserva 10% dos dados de treino (os mais recentes, pois estão ordenados por data) como validação para parar automaticamente quando não há melhora por 30 iterações.

In [ ]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=1000,
    max_depth=5,
    min_samples_leaf=12,
    l2_regularization=0.2,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,
    random_state=42,
)
hgb.fit(X_train, y_train, sample_weight=sample_weight)
print(f"Iterações efetivas (early stopping): {hgb.n_iter_}")

## Seção 6 — Cross-Validation

`TimeSeriesSplit(n_splits=5)` respeita a ordem temporal: cada fold usa apenas dados passados para treinar e dados futuros para validar. Os pesos de recência são aplicados em cada fold de treino separadamente.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_acc, cv_f1 = [], []

for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    w_tr = sample_weight[tr_idx]

    m = HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=500, max_depth=5,
        min_samples_leaf=12, l2_regularization=0.2,
        early_stopping=True, validation_fraction=0.1,
        n_iter_no_change=30, random_state=42,
    )
    m.fit(X_tr, y_tr, sample_weight=w_tr)
    preds = m.predict(X_val)
    cv_acc.append(accuracy_score(y_val, preds))
    cv_f1.append(f1_score(y_val, preds, average="macro"))
    print(f"  Fold {fold}: acc={cv_acc[-1]:.3f}  macro-F1={cv_f1[-1]:.3f}")

print(f"\nCV Accuracy : {np.mean(cv_acc):.3f} ± {np.std(cv_acc):.3f}")
print(f"CV Macro-F1 : {np.mean(cv_f1):.3f} ± {np.std(cv_f1):.3f}")

## Seção 7 — Avaliação no Teste

O conjunto de teste é tocado **uma única vez**, aqui. Qualquer ajuste de hiperparâmetros baseado neste resultado seria data leakage.

In [ ]:
y_pred = hgb.predict(X_test)

print(classification_report(
    y_test, y_pred,
    labels=["HomeWin", "Draw", "AwayWin"],
    digits=3,
))

cm = confusion_matrix(y_test, y_pred, labels=["HomeWin", "Draw", "AwayWin"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["HomeWin", "Draw", "AwayWin"],
    yticklabels=["HomeWin", "Draw", "AwayWin"],
    ax=ax,
)
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("Confusion Matrix — Conjunto de Teste")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=120)
plt.show()

## Seção 8 — Baseline Comparison

In [ ]:
acc       = accuracy_score(y_test, y_pred)
macro_f1  = f1_score(y_test, y_pred, average="macro")
naive_acc = 0.456

print(f"{'Método':<30} {'Accuracy':>10} {'Macro-F1':>10}")
print("-" * 52)
print(f"{'Naive (sempre HomeWin)':<30} {naive_acc:>10.1%} {'—':>10}")
print(f"{'HistGradientBoosting':<30} {acc:>10.1%} {macro_f1:>10.3f}")
print(f"\nGanho sobre baseline: +{(acc - naive_acc):.1%}")

## Seção 9 — `predict_match()`

Função de inferência ad-hoc: recebe dois nomes de times e retorna a predição do modelo usando as features mais recentes disponíveis de cada time no dataset combinado (treino + teste).

In [ ]:
all_df = pd.concat([train, test], ignore_index=True).sort_values("Date").reset_index(drop=True)


def predict_match(home_team_name: str, away_team_name: str) -> str:
    all_teams = sorted(set(all_df["home_team"]) | set(all_df["away_team"]))

    def _resolve(name: str) -> str:
        if name in all_teams:
            return name
        suggestions = difflib.get_close_matches(name, all_teams, n=3, cutoff=0.6)
        suffix = f" Você quis dizer: {suggestions}?" if suggestions else ""
        raise ValueError(f"Time '{name}' não encontrado.{suffix}")

    h = _resolve(home_team_name)
    a = _resolve(away_team_name)

    def _last_row_and_role(team: str):
        candidates = []
        home_rows = all_df[all_df["home_team"] == team]
        away_rows = all_df[all_df["away_team"] == team]
        if len(home_rows):
            r = home_rows.loc[home_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "home"))
        if len(away_rows):
            r = away_rows.loc[away_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "away"))
        if not candidates:
            raise ValueError(f"Sem dados históricos para '{team}'")
        _, last, role = max(candidates, key=lambda x: x[0])
        return last, role

    h_row, h_role = _last_row_and_role(h)
    a_row, a_role = _last_row_and_role(a)

    # Stats de forma por time (independente do papel em cada partida)
    team_stats = [
        "goals_scored_last5", "goals_conceded_last5",
        "wins_last5", "draws_last5", "losses_last5",
        "goal_diff_last5", "points_last5",
        "shots_on_target_last5", "corners_last5",
        "win_pct_season", "draw_pct_season", "loss_pct_season",
        "ppg_season", "goal_eff",
    ]

    feat = {}
    for s in team_stats:
        feat[f"home_{s}"] = h_row.get(f"{h_role}_{s}", 0.0)
        feat[f"away_{s}"] = a_row.get(f"{a_role}_{s}", 0.0)

    # Features específicas de local (última aparição naquele papel)
    h_home = all_df[all_df["home_team"] == h].sort_values("Date")
    a_away = all_df[all_df["away_team"] == a].sort_values("Date")
    feat["home_wins_at_home_last5"] = float(h_home["home_wins_at_home_last5"].iloc[-1]) if len(h_home) else 0.0
    feat["away_wins_away_last5"]    = float(a_away["away_wins_away_last5"].iloc[-1])    if len(a_away) else 0.0

    # Elo (pré-partida do último jogo do time)
    feat["elo_home"] = float(h_row[f"elo_{h_role}"])
    feat["elo_away"] = float(a_row[f"elo_{a_role}"])
    feat["elo_diff"] = feat["elo_home"] - feat["elo_away"]

    # Probs implícitas (sem odds ao vivo → média global como prior)
    feat["implied_home_prob"] = float(all_df["implied_home_prob"].mean())
    feat["implied_draw_prob"] = float(all_df["implied_draw_prob"].mean())
    feat["implied_away_prob"] = float(all_df["implied_away_prob"].mean())

    # Taxas históricas
    feat["home_team_hist_wr"] = float(h_home["home_team_hist_wr"].iloc[-1]) if len(h_home) else float(all_df["home_team_hist_wr"].mean())
    feat["away_team_hist_wr"] = float(a_away["away_team_hist_wr"].iloc[-1]) if len(a_away) else float(all_df["away_team_hist_wr"].mean())

    # Fase da temporada: meio de temporada como fallback
    feat["round_norm"] = 0.5

    # Vantagem relativa (recalculada)
    feat["goals_scored_adv"]   = feat["home_goals_scored_last5"]    - feat["away_goals_scored_last5"]
    feat["goals_conceded_adv"] = feat["away_goals_conceded_last5"]  - feat["home_goals_conceded_last5"]
    feat["points_adv"]         = feat["home_points_last5"]          - feat["away_points_last5"]
    feat["goal_diff_adv"]      = feat["home_goal_diff_last5"]       - feat["away_goal_diff_last5"]
    feat["win_pct_adv"]        = feat["home_win_pct_season"]        - feat["away_win_pct_season"]
    feat["shots_adv"]          = feat["home_shots_on_target_last5"] - feat["away_shots_on_target_last5"]
    feat["ppg_adv"]            = feat["home_ppg_season"]            - feat["away_ppg_season"]
    feat["goal_eff_adv"]       = feat["home_goal_eff"]              - feat["away_goal_eff"]

    X_inf = pd.DataFrame([feat])[FEATURE_COLS]
    pred  = hgb.predict(X_inf)[0]
    proba = dict(zip(hgb.classes_, hgb.predict_proba(X_inf)[0].round(3)))

    print(f"{h} (casa)  vs  {a} (fora)")
    print(f"Predição    : {pred}")
    print(f"Probabilidades: {proba}")
    return pred

In [ ]:
# Teste com exemplo real
_ = predict_match("Arsenal", "Chelsea")
print()

# Teste com nome inválido — deve sugerir o nome correto
try:
    predict_match("Arsnel", "Chelsea")
except ValueError as e:
    print(f"Erro esperado: {e}")

## Seção 10 — Interpretação dos Resultados

### Resultados obtidos

| Métrica | Valor |
|---------|------:|
| **Accuracy (teste)** | **51,8 %** |
| Baseline ingênuo (sempre HomeWin) | 45,6 % |
| Ganho sobre baseline | **+6,2 pp** |
| **Macro-F1 (teste)** | **0,405** |
| CV Accuracy (TimeSeriesSplit × 5) | 52,1 % ± 1,5 % |
| CV Macro-F1 | 0,432 ± 0,021 |

---

### Recall por classe

| Classe | Recall | Interpretação |
|--------|-------:|---------------|
| HomeWin | 0,746 | O modelo identifica bem a maioria das vitórias do mandante |
| AwayWin | 0,565 | Desempenho razoável nas vitórias do visitante |
| Draw | 0,036 | Empates são muito difíceis de prever — classe com menos sinal |

O baixo recall em *Draw* é esperado e universal em modelos de futebol: empates são intrinsecamente ruidosos, sem correlação forte com features pré-jogo. A maioria dos modelos da literatura simplesmente aprende a não predizer esta classe.

---

### Contexto e teto realista

O **teto realista para previsão de futebol com dados abertos** (sem dados de tracking, escalações ao vivo ou odds em tempo real) é estimado entre **54–58 % de accuracy** nos melhores trabalhos da literatura. Modelos baseados exclusivamente em form e Elo tipicamente atingem **51–54 %**.

O resultado de **51,8 %** está dentro do esperado para este conjunto de features. Para avançar além deste patamar, as principais alavancas são:

1. **Odds ao vivo** como feature de inferência (não apenas históricas)
2. **Dados de escalação e lesões** — ausência de titulares é o maior preditor individual
3. **Expected Goals (xG)** — mais informativo que gols marcados para estimar forma real
4. **Calibração das probabilidades** — `CalibratedClassifierCV` para melhorar as saídas de `predict_proba`

## Seção 11 — Predição Seletiva por Confiança

Em vez de forçar uma predição para cada partida, o modelo **abstém** quando a probabilidade máxima não supera um limiar de confiança `θ`.

```
confiança = max(P(HomeWin), P(Draw), P(AwayWin))
se confiança ≥ θ → retorna predição
se confiança <  θ → abstém
```

**Trade-off:** quanto maior o `θ`, menos partidas passam o filtro (*cobertura* cai), mas a *precisão* nas que passam aumenta. O objetivo aqui não é accuracy global — é ter alta confiança nas poucas partidas que o modelo escolhe responder.

In [ ]:
# Confiança = probabilidade máxima do modelo principal (treinado em 2005–2021)
proba_test_full = hgb.predict_proba(X_test)
classes_order   = list(hgb.classes_)

confidence    = proba_test_full.max(axis=1)
predicted_cls = np.array([classes_order[i] for i in proba_test_full.argmax(axis=1)])
correct       = predicted_cls == y_test.values

# Curva cobertura × precisão para limiares de 0.45 a 0.90
thresholds  = np.arange(0.45, 0.91, 0.01)
coverages, precisions, counts = [], [], []

for θ in thresholds:
    mask = confidence >= θ
    n    = mask.sum()
    counts.append(n)
    coverages.append(n / len(confidence))
    precisions.append(correct[mask].mean() if n > 0 else np.nan)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()

ax1.plot(thresholds, precisions, color="#2196F3", lw=2, label="Precisão (acertos / filtradas)")
ax2.plot(thresholds, coverages,  color="#FF9800", lw=2, linestyle="--", label="Cobertura (% partidas)")

ax1.axhline(0.456, color="gray", linestyle=":", lw=1, label="Baseline naive (45.6%)")
ax1.axvline(0.85,  color="red",  linestyle="--", lw=1, label="θ = 0.85")

ax1.set_xlabel("Limiar de confiança (θ)")
ax1.set_ylabel("Precisão", color="#2196F3")
ax2.set_ylabel("Cobertura", color="#FF9800")
ax1.set_ylim(0.40, 1.05)
ax2.set_ylim(0, 1.05)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)
ax1.set_title("Predição seletiva — Precisão vs. Cobertura (conjunto de teste)")
plt.tight_layout()
plt.savefig("selective_prediction_curve.png", dpi=120)
plt.show()

# Tabela de referência
print(f"\n{'θ':>6}  {'Partidas':>9}  {'Cobertura':>10}  {'Precisão':>10}")
print("-" * 42)
for θ in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    mask = confidence >= θ
    n    = mask.sum()
    prec = correct[mask].mean() if n > 0 else float("nan")
    cov  = n / len(confidence)
    print(f"{θ:>6.2f}  {n:>9}  {cov:>10.1%}  {prec:>10.1%}")

In [ ]:
def predict_match_selective(
    home_team_name: str,
    away_team_name: str,
    confidence_threshold: float = 0.70,
) -> dict | None:
    """
    Retorna predição apenas se a confiança (max proba) >= threshold.
    Caso contrário retorna None — o modelo abstém.
    """
    all_teams = sorted(set(all_df["home_team"]) | set(all_df["away_team"]))

    def _resolve(name: str) -> str:
        if name in all_teams:
            return name
        suggestions = difflib.get_close_matches(name, all_teams, n=3, cutoff=0.6)
        suffix = f" Você quis dizer: {suggestions}?" if suggestions else ""
        raise ValueError(f"Time '{name}' não encontrado.{suffix}")

    h = _resolve(home_team_name)
    a = _resolve(away_team_name)

    def _last_row_and_role(team):
        candidates = []
        home_rows = all_df[all_df["home_team"] == team]
        away_rows = all_df[all_df["away_team"] == team]
        if len(home_rows):
            r = home_rows.loc[home_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "home"))
        if len(away_rows):
            r = away_rows.loc[away_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "away"))
        _, last, role = max(candidates, key=lambda x: x[0])
        return last, role

    h_row, h_role = _last_row_and_role(h)
    a_row, a_role = _last_row_and_role(a)

    team_stats = [
        "goals_scored_last5", "goals_conceded_last5",
        "wins_last5", "draws_last5", "losses_last5",
        "goal_diff_last5", "points_last5",
        "shots_on_target_last5", "corners_last5",
        "win_pct_season", "draw_pct_season", "loss_pct_season",
        "ppg_season", "goal_eff",
    ]

    feat = {}
    for s in team_stats:
        feat[f"home_{s}"] = h_row.get(f"{h_role}_{s}", 0.0)
        feat[f"away_{s}"] = a_row.get(f"{a_role}_{s}", 0.0)

    h_home = all_df[all_df["home_team"] == h].sort_values("Date")
    a_away = all_df[all_df["away_team"] == a].sort_values("Date")
    feat["home_wins_at_home_last5"] = float(h_home["home_wins_at_home_last5"].iloc[-1]) if len(h_home) else 0.0
    feat["away_wins_away_last5"]    = float(a_away["away_wins_away_last5"].iloc[-1])    if len(a_away) else 0.0
    feat["elo_home"] = float(h_row[f"elo_{h_role}"])
    feat["elo_away"] = float(a_row[f"elo_{a_role}"])
    feat["elo_diff"] = feat["elo_home"] - feat["elo_away"]
    feat["implied_home_prob"] = float(all_df["implied_home_prob"].mean())
    feat["implied_draw_prob"] = float(all_df["implied_draw_prob"].mean())
    feat["implied_away_prob"] = float(all_df["implied_away_prob"].mean())
    feat["home_team_hist_wr"] = float(h_home["home_team_hist_wr"].iloc[-1]) if len(h_home) else float(all_df["home_team_hist_wr"].mean())
    feat["away_team_hist_wr"] = float(a_away["away_team_hist_wr"].iloc[-1]) if len(a_away) else float(all_df["away_team_hist_wr"].mean())
    feat["round_norm"]        = 0.5
    feat["goals_scored_adv"]   = feat["home_goals_scored_last5"]    - feat["away_goals_scored_last5"]
    feat["goals_conceded_adv"] = feat["away_goals_conceded_last5"]  - feat["home_goals_conceded_last5"]
    feat["points_adv"]         = feat["home_points_last5"]          - feat["away_points_last5"]
    feat["goal_diff_adv"]      = feat["home_goal_diff_last5"]       - feat["away_goal_diff_last5"]
    feat["win_pct_adv"]        = feat["home_win_pct_season"]        - feat["away_win_pct_season"]
    feat["shots_adv"]          = feat["home_shots_on_target_last5"] - feat["away_shots_on_target_last5"]
    feat["ppg_adv"]            = feat["home_ppg_season"]            - feat["away_ppg_season"]
    feat["goal_eff_adv"]       = feat["home_goal_eff"]              - feat["away_goal_eff"]

    X_inf = pd.DataFrame([feat])[FEATURE_COLS]
    proba = hgb.predict_proba(X_inf)
    conf  = float(proba.max())
    pred  = hgb.classes_[int(proba.argmax())]
    proba_dict = dict(zip(hgb.classes_, proba[0].round(3)))

    if conf < confidence_threshold:
        print(f"{h} vs {a}  →  ABSTÉM  (confiança {conf:.1%} < θ={confidence_threshold:.0%})")
        return None

    print(f"{h} (casa) vs {a} (fora)")
    print(f"  Predição   : {pred}")
    print(f"  Confiança  : {conf:.1%}")
    print(f"  Probabilidades: {proba_dict}")
    return {"prediction": pred, "confidence": conf, "proba": proba_dict}


# Demonstração
jogos = [
    ("Man City",  "Sheffield United"),
    ("Arsenal",   "Chelsea"),
    ("Liverpool", "Everton"),
    ("Burnley",   "Man City"),
]

print("=== Predição seletiva (θ = 70%) ===\n")
for home, away in jogos:
    predict_match_selective(home, away, confidence_threshold=0.70)
    print()

## Seção 12 — Kelly Criterion: Tamanho Ótimo de Aposta

Em vez de apostar um valor fixo, o **Critério de Kelly** determina a fração ideal do bankroll a apostar em cada jogo, ponderando a confiança do modelo com as odds da casa:

```
f = (p × odd − 1) / (odd − 1)
```

- `p` = probabilidade estimada pelo modelo para o resultado predito
- `odd` = odds da Bet365 para esse resultado
- `f > 0` → apostar `f × bankroll` (o modelo tem edge sobre a casa)
- `f ≤ 0` → não apostar (a casa está com edge)

Naturalmente, jogos onde o modelo é menos confiante que as odds implícitas da casa são descartados. Usamos duas variantes:
- **Full Kelly** — fração máxima teórica (mais agressivo)
- **Half Kelly** — metade da fração (mais conservador, reduz variância)

Bankroll inicial: **R$ 10.000** | Cap máximo por aposta: **25% do bankroll**

In [ ]:
BANKROLL_INIT = 10_000
MAX_FRACTION  = 0.25

test_sorted  = test.sort_values("Date").reset_index(drop=True)
proba_all    = hgb.predict_proba(test_sorted[FEATURE_COLS])
classes_list = list(hgb.classes_)

# Backtest Kelly para diferentes filtros de confiança mínima
def run_kelly(min_conf: float, half: bool = True, label: str = ""):
    bk = BANKROLL_INIT
    records = []
    mult = 0.5 if half else 1.0

    for i, row in test_sorted.iterrows():
        p_vec    = proba_all[i]
        pred_idx = int(np.argmax(p_vec))
        pred     = classes_list[pred_idx]
        p        = float(p_vec[pred_idx])
        conf     = p  # confiança = probabilidade da classe predita

        odd = row["B365H"] if pred == "HomeWin" else (row["B365A"] if pred == "AwayWin" else row["B365D"])
        b   = odd - 1
        kelly = (p * b - (1 - p)) / b

        correct = row["result"] == pred

        # Só aposta se kelly > 0 E confiança >= limiar
        if kelly > 0 and conf >= min_conf:
            stake  = min(kelly * mult * bk, MAX_FRACTION * bk)
            profit = stake * b if correct else -stake
            bk    += profit
        else:
            stake  = 0.0
            profit = 0.0

        records.append({
            "Date": row["Date"], "Season": row["Season"],
            "predicted": pred, "result": row["result"],
            "p": p, "odd": odd, "kelly": kelly,
            "correct": correct, "stake": stake, "profit": profit, "bankroll": bk,
        })

    dk = pd.DataFrame(records)
    bet = dk[dk["stake"] > 0]
    roi = (bk / BANKROLL_INIT - 1) * 100
    return dk, bet, bk, roi

# Rodar para diferentes thresholds
configs = [
    (0.00, False, "Full Kelly puro"),
    (0.00, True,  "Half Kelly puro"),
    (0.70, True,  "Half Kelly + θ≥70%"),
    (0.75, True,  "Half Kelly + θ≥75%"),
    (0.80, True,  "Half Kelly + θ≥80%"),
]

resultados = []
for min_conf, half, label in configs:
    dk, bet, bk_final, roi = run_kelly(min_conf, half, label)
    resultados.append((label, len(bet), bet["correct"].mean() if len(bet) > 0 else 0,
                       bk_final, roi, dk))
    print(f"{label:<30} | {len(bet):4d} apostas | {bet['correct'].mean():.1%} acertos | "
          f"R${bk_final:>9,.0f} | ROI {roi:>+7.1f}%")

print()
print("Por temporada — Half Kelly + θ≥75%:")
_, bet75, _, _ = run_kelly(0.75, True)
dk75, bet75_df, _, _ = run_kelly(0.75, True)
for s, g in dk75[dk75["stake"] > 0].groupby("Season"):
    print(f"  {s}: {len(g):3d} apostas | {g['correct'].sum():3d} acertos ({g['correct'].mean():.0%}) | "
          f"Bankroll final: R${g['bankroll'].iloc[-1]:,.0f}")

In [ ]:
# Gráfico: evolução do bankroll — Kelly filtrado vs fixed stake
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

dk_kelly75, bet_k75, bk_k75, roi_k75 = run_kelly(0.75, True)
dk_kelly80, bet_k80, bk_k80, roi_k80 = run_kelly(0.80, True)

ax1 = axes[0]
ax1.plot(dk_kelly75["Date"], dk_kelly75["bankroll"], color="#2196F3", lw=2,
         label=f"Half Kelly + θ≥75%  (ROI {roi_k75:+.1f}%)")
ax1.plot(dk_kelly80["Date"], dk_kelly80["bankroll"], color="#4CAF50", lw=2,
         label=f"Half Kelly + θ≥80%  (ROI {roi_k80:+.1f}%)")
ax1.axhline(BANKROLL_INIT, color="gray", lw=1, linestyle="--", label="Bankroll inicial R$10.000")
ax1.set_ylabel("Bankroll (R$)")
ax1.set_title("Evolução do Bankroll — Half Kelly + filtro de confiança")
ax1.legend(fontsize=9)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"R${x:,.0f}"))

# Painel 2: stake por jogo (Half Kelly + θ≥75%)
ax2 = axes[1]
bet_only = dk_kelly75[dk_kelly75["stake"] > 0]
colors_c = ["#2196F3" if c else "#F44336" for c in bet_only["correct"]]
ax2.bar(bet_only["Date"], bet_only["stake"], color=colors_c, alpha=0.6, width=3)
ax2.set_ylabel("Valor apostado (R$)")
ax2.set_xlabel("Data")
ax2.set_title("Half Kelly + θ≥75% — valor por jogo (azul=acerto, vermelho=erro)")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"R${x:,.0f}"))

plt.tight_layout()
plt.savefig("grafico_kelly_bankroll.png", dpi=120)
plt.show()

# Tabela comparativa final completa
print(f"\n{'='*70}")
print(f"{'COMPARATIVO — Partindo de R$10.000':^70}")
print(f"{'='*70}")
print(f"{'Estratégia':<35} {'Jogos':>6} {'Acertos':>8} {'ROI':>8} {'Lucro':>12}")
print(f"{'-'*70}")

# Fixed stake (simulado com R$1.000/aposta)
proba_t = hgb.predict_proba(test[FEATURE_COLS])
conf_t  = proba_t.max(axis=1)
pred_t  = np.array([classes_list[i] for i in proba_t.argmax(axis=1)])
for θ, lbl in [(0.75, "Fixed R$1.000  θ≥75%"), (0.80, "Fixed R$1.000  θ≥80%")]:
    m = conf_t >= θ
    df_f = test[m][["result","B365H","B365D","B365A"]].copy()
    df_f["pred"] = pred_t[m]
    df_f["correct"] = df_f["result"] == df_f["pred"]
    df_f["odd"] = df_f.apply(lambda r: r["B365H"] if r["pred"]=="HomeWin" else r["B365A"], axis=1)
    df_f["lucro"] = df_f.apply(lambda r: (r["odd"]-1)*1000 if r["correct"] else -1000, axis=1)
    print(f"{lbl:<35} {len(df_f):>6} {df_f['correct'].mean():>7.1%} {df_f['lucro'].sum()/(len(df_f)*1000)*100:>+7.1f}%  R$ {df_f['lucro'].sum():>+9,.0f}")

for lbl, n, acc, bk, roi, _ in resultados:
    if "puro" not in lbl:
        lucro = bk - BANKROLL_INIT
        print(f"{lbl:<35} {n:>6} {acc:>7.1%} {roi:>+7.1f}%  R$ {lucro:>+9,.0f}")

## Seção 13 — `predict_match_kelly()`

Função de inferência que combina predição seletiva com dimensionamento de aposta via Critério de Kelly.

Recebe o nome dos times e a **odd real** do jogo (necessária para calcular o Kelly) e retorna:
- A predição (`HomeWin`, `Draw` ou `AwayWin`)
- A confiança do modelo
- A fração do bankroll a apostar (Half Kelly)
- O valor em reais a apostar, dado um bankroll informado

Se a confiança estiver abaixo de θ ou o Kelly for negativo (casa com edge), a função **abstém**.

In [ ]:
def predict_match_kelly(
    home_team_name: str,
    away_team_name: str,
    odd_home: float,
    odd_draw: float,
    odd_away: float,
    bankroll: float,
    confidence_threshold: float = 0.75,
    max_fraction: float = 0.25,
) -> dict | None:
    """
    Prediz o resultado e calcula quanto apostar via Half Kelly.

    Parâmetros
    ----------
    home_team_name / away_team_name : nomes dos times
    odd_home / odd_draw / odd_away  : odds decimais atuais da casa (ex: 1.40, 3.20, 2.80)
    bankroll                        : seu bankroll atual em R$
    confidence_threshold            : confiança mínima para apostar (default 0.75)
    max_fraction                    : cap máximo da fração do bankroll (default 25%)

    Retorna None se o modelo abstém (baixa confiança ou edge negativo).
    """
    team_stats = [
        "goals_scored_last5", "goals_conceded_last5",
        "wins_last5", "draws_last5", "losses_last5",
        "goal_diff_last5", "points_last5",
        "shots_on_target_last5", "corners_last5",
        "win_pct_season", "draw_pct_season", "loss_pct_season",
        "ppg_season", "goal_eff",
    ]

    all_teams = sorted(set(all_df["home_team"]) | set(all_df["away_team"]))

    def _resolve(name: str) -> str:
        if name in all_teams:
            return name
        suggestions = difflib.get_close_matches(name, all_teams, n=3, cutoff=0.6)
        suffix = f" Você quis dizer: {suggestions}?" if suggestions else ""
        raise ValueError(f"Time '{name}' não encontrado.{suffix}")

    h = _resolve(home_team_name)
    a = _resolve(away_team_name)

    def _last_row_and_role(team):
        candidates = []
        home_rows = all_df[all_df["home_team"] == team]
        away_rows = all_df[all_df["away_team"] == team]
        if len(home_rows):
            r = home_rows.loc[home_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "home"))
        if len(away_rows):
            r = away_rows.loc[away_rows["Date"].idxmax()]
            candidates.append((r["Date"], r, "away"))
        _, last, role = max(candidates, key=lambda x: x[0])
        return last, role

    h_row, h_role = _last_row_and_role(h)
    a_row, a_role = _last_row_and_role(a)

    feat = {}
    for s in team_stats:
        feat[f"home_{s}"] = h_row.get(f"{h_role}_{s}", 0.0)
        feat[f"away_{s}"] = a_row.get(f"{a_role}_{s}", 0.0)

    h_home = all_df[all_df["home_team"] == h].sort_values("Date")
    a_away = all_df[all_df["away_team"] == a].sort_values("Date")
    feat["home_wins_at_home_last5"] = float(h_home["home_wins_at_home_last5"].iloc[-1]) if len(h_home) else 0.0
    feat["away_wins_away_last5"]    = float(a_away["away_wins_away_last5"].iloc[-1])    if len(a_away) else 0.0
    feat["elo_home"] = float(h_row[f"elo_{h_role}"])
    feat["elo_away"] = float(a_row[f"elo_{a_role}"])
    feat["elo_diff"] = feat["elo_home"] - feat["elo_away"]

    # Odds reais fornecidas — usadas tanto para implied probs quanto para Kelly
    total_inv = 1/odd_home + 1/odd_draw + 1/odd_away
    feat["implied_home_prob"] = (1/odd_home) / total_inv
    feat["implied_draw_prob"] = (1/odd_draw) / total_inv
    feat["implied_away_prob"] = (1/odd_away) / total_inv

    feat["home_team_hist_wr"] = float(h_home["home_team_hist_wr"].iloc[-1]) if len(h_home) else float(all_df["home_team_hist_wr"].mean())
    feat["away_team_hist_wr"] = float(a_away["away_team_hist_wr"].iloc[-1]) if len(a_away) else float(all_df["away_team_hist_wr"].mean())
    feat["round_norm"]        = 0.5
    feat["goals_scored_adv"]   = feat["home_goals_scored_last5"]    - feat["away_goals_scored_last5"]
    feat["goals_conceded_adv"] = feat["away_goals_conceded_last5"]  - feat["home_goals_conceded_last5"]
    feat["points_adv"]         = feat["home_points_last5"]          - feat["away_points_last5"]
    feat["goal_diff_adv"]      = feat["home_goal_diff_last5"]       - feat["away_goal_diff_last5"]
    feat["win_pct_adv"]        = feat["home_win_pct_season"]        - feat["away_win_pct_season"]
    feat["shots_adv"]          = feat["home_shots_on_target_last5"] - feat["away_shots_on_target_last5"]
    feat["ppg_adv"]            = feat["home_ppg_season"]            - feat["away_ppg_season"]
    feat["goal_eff_adv"]       = feat["home_goal_eff"]              - feat["away_goal_eff"]

    X_inf        = pd.DataFrame([feat])[FEATURE_COLS]
    proba        = hgb.predict_proba(X_inf)[0]
    classes_ord  = list(hgb.classes_)
    pred_idx     = int(np.argmax(proba))
    pred         = classes_ord[pred_idx]
    p            = float(proba[pred_idx])

    if p < confidence_threshold:
        print(f"{h} vs {a}  →  ABSTÉM  (confiança {p:.1%} < θ={confidence_threshold:.0%})")
        return None

    odd     = {"HomeWin": odd_home, "Draw": odd_draw, "AwayWin": odd_away}[pred]
    b       = odd - 1
    kelly   = (p * b - (1 - p)) / b
    k_half  = kelly * 0.5

    if k_half <= 0:
        print(f"{h} vs {a}  →  ABSTÉM  (Kelly negativo: edge insuficiente vs. odds da casa)")
        return None

    fraction = min(k_half, max_fraction)
    stake    = round(fraction * bankroll, 2)

    print(f"{'─'*52}")
    print(f"{h} (casa)  vs  {a} (fora)")
    print(f"  Predição      : {pred}")
    print(f"  Confiança     : {p:.1%}")
    print(f"  Odd fornecida : {odd:.2f}  (lucro líquido: {b:.2f}×)")
    print(f"  Kelly         : {kelly:.3f}  →  Half Kelly: {k_half:.3f}")
    print(f"  Bankroll      : R${bankroll:,.2f}")
    print(f"  Apostar       : R${stake:,.2f}  ({fraction:.1%} do bankroll)")
    print(f"  Se acertar    : +R${round(stake*b,2):,.2f}")
    print(f"  Se errar      : −R${stake:,.2f}")
    print(f"{'─'*52}")

    return {"prediction": pred, "confidence": p, "odd": odd,
            "kelly": kelly, "kelly_half": k_half,
            "fraction": fraction, "stake": stake,
            "profit_if_correct": round(stake * b, 2),
            "loss_if_wrong": stake}


# ── Exemplos de uso ───────────────────────────────────────────────────────────
_ = predict_match_kelly(
    "Man City", "Southampton",
    odd_home=1.30, odd_draw=5.50, odd_away=9.00,
    bankroll=10_000, confidence_threshold=0.75,
)
print()
_ = predict_match_kelly(
    "Arsenal", "Chelsea",
    odd_home=1.80, odd_draw=3.60, odd_away=4.50,
    bankroll=10_000, confidence_threshold=0.75,
)